# Chapter 5 companion: realizations, state variables, and the sample-and-hold.

*Computers in the Loop: Digital Control from Sampling to Networked Agents*: EECE 5610 lecture notes prepared and revised by Milad Siami.

Course sources: Mario Sznaier's lecture notes; T. Charalambous's slides; Phillips, Nagle and Chakrabortty, *Digital Control System Analysis and Design*, 4th ed., Pearson (2015).

Reproduces:
  * G(z) = (z + 0.5)/(z^2 - 0.75z + 0.125) run three ways from rest with a unit
    step: the direct form (four delays), the two-delay form, and the state
    equations x(k+1) = A x(k) + B e(k), m(k) = C x(k); all give
    m_k = 4 (1 - 0.5^k)^2;
  * G(z) = C (zI - A)^-1 B evaluated at test points;
  * the sampled-signal table on E(s) = 1/((s+1)(s+2)) with T = ln 2:
    E(z) = 0.25 z/((z - 0.5)(z - 0.25));
  * a ramp through the sample-and-hold, and the hold-error bound T max|de/dt|.

Run the cells from top to bottom (in Colab: *Runtime > Run all*). Figures are also saved to a `figures` folder.


In [ ]:
from pathlib import Path

import numpy as np
import matplotlib
import matplotlib.pyplot as plt
from scipy import signal

FIG = Path.cwd() / "figures"
FIG.mkdir(exist_ok=True)

b0, b1 = 1.0, 0.5                 # numerator  beta0 z + beta1
a1, a2 = -0.75, 0.125             # denominator z^2 + alpha1 z + alpha2
N = 13
e = np.ones(N)

In [ ]:
# direct form: m_k = b0 e_{k-1} + b1 e_{k-2} - a1 m_{k-1} - a2 m_{k-2}
m_dir = np.zeros(N)

In [ ]:
for k in range(N):
    em1 = e[k - 1] if k >= 1 else 0.0
    em2 = e[k - 2] if k >= 2 else 0.0
    mm1 = m_dir[k - 1] if k >= 1 else 0.0
    mm2 = m_dir[k - 2] if k >= 2 else 0.0
    m_dir[k] = b0 * em1 + b1 * em2 - a1 * mm1 - a2 * mm2

In [ ]:
# two-delay form: x2(k+1) = e_k - a1 x2 - a2 x1, x1(k+1) = x2, m = x2 + 0.5 x1
x1 = x2 = 0.0

In [ ]:
m_two, X1, X2 = [], [], []

In [ ]:
for k in range(N):
    m_two.append(b0 * x2 + b1 * x1)
    X1.append(x1)
    X2.append(x2)
    x1, x2 = x2, e[k] - a1 * x2 - a2 * x1

In [ ]:
# state equations (the same two delays as matrices)
A = np.array([[0.0, 1.0], [-a2, -a1]])

In [ ]:
B = np.array([[0.0], [1.0]])
C = np.array([[b1, b0]])
x = np.zeros((2, 1))
m_ss = []

In [ ]:
for k in range(N):
    m_ss.append((C @ x).item())
    x = A @ x + B * e[k]

In [ ]:
closed = 4 * (1 - 0.5 ** np.arange(N)) ** 2
print("G(z) = (z + 0.5)/(z^2 - 0.75z + 0.125), unit step from rest")
print("  x1(k)            :", np.round(X1[:6], 4))
print("  x2(k)            :", np.round(X2[:6], 4))
print("  direct form      :", np.round(m_dir[:6], 4))
print("  two-delay form   :", np.round(m_two[:6], 4))
print("  state equations  :", np.round(m_ss[:6], 4))
print("  4 (1 - 0.5^k)^2  :", np.round(closed[:6], 4))
print("  final value G(1) =", (1 + 0.5) / (1 - 0.75 + 0.125))
r, p, _ = signal.residue([1, 0.5], np.polymul([1, a1, a2], [1, -1]))
print("  residues of M(z)/z at", np.round(p, 4), ":", np.round(r, 4))

In [ ]:
for z in (2.0, 1.1 + 0.9j):
    G_ss = (C @ np.linalg.solve(z * np.eye(2) - A, B)).item()
    print(f"  z = {z}: C (zI - A)^-1 B = {complex(G_ss):.6f},  G(z) = {complex((z + 0.5) / (z * z - 0.75 * z + 0.125)):.6f}")

In [ ]:
print("  det(zI - A) coefficients:", np.round(np.poly(A), 4))

In [ ]:
# ---------------------------------------------------------------- sampled-signal table
T = np.log(2)

In [ ]:
k = np.arange(8)
samples = np.exp(-k * T) - np.exp(-2 * k * T)
from_table = signal.lfilter([0, 0.25, 0], np.polymul([1, -0.5], [1, -0.25]), np.r_[1.0, np.zeros(7)])
print("\nE(s) = 1/((s+1)(s+2)), T = ln 2")
print("  samples e(kT)              :", np.round(samples, 5))
print("  series of 0.25z/((z-0.5)(z-0.25)):", np.round(from_table, 5))

In [ ]:
# ---------------------------------------------------------------- ramp through the hold
T = 0.5

In [ ]:
t = np.linspace(0, 3, 3001)
held = T * np.floor(t / T + 1e-12)
gap = t - held
print(f"\nRamp through the ZOH, T = {T}: gap e(t) - ebar(t) between {gap.min():.3f} and {gap.max():.3f},"
      f" mean {gap.mean():.3f} (about T/2 = {T / 2})")
print(f"Hold error for sin(2 pi t) below 0.05 needs T <= 0.05/(2 pi) = {0.05 / (2 * np.pi):.5f} s"
      f" ({1 / (0.05 / (2 * np.pi)):.0f} samples per cycle)")
Tq = 0.05 / (2 * np.pi)
tt = np.linspace(0, 1, 200001)
err = np.abs(np.sin(2 * np.pi * tt) - np.sin(2 * np.pi * Tq * np.floor(tt / Tq)))
print(f"  largest error with that T: {err.max():.4f}")

In [ ]:
# ---------------------------------------------------------------- figure
fig, axes = plt.subplots(1, 2, figsize=(10, 3.4))

In [ ]:
axes[0].stem(np.arange(N), m_dir)
axes[0].axhline(4, ls="--", color="gray", lw=0.8)
axes[0].set(xlabel="k", ylabel="m_k", title="Step response of G(z): every realization")
tt = np.linspace(0, 4, 800)
Ts = 0.5
axes[1].plot(tt, 1 - np.exp(-tt), label="e(t) = 1 - e^-t")
axes[1].step(tt, 1 - np.exp(-Ts * np.floor(tt / Ts)), where="post", label="held, T = 0.5 s")
axes[1].plot(Ts * np.arange(9), 1 - np.exp(-Ts * np.arange(9)), "o", ms=4, label="same samples")
axes[1].set(xlabel="t (s)", title="Same samples, different signals")
axes[1].legend(fontsize=8)
fig.tight_layout()
fig.savefig(FIG / "ch05_realizations.png", dpi=150)
print("\nSaved figures/ch05_realizations.png")

## Explore




In [ ]:
# In the browser (JupyterLite) the sliders need one extra package.
import sys
if "pyodide" in sys.modules:
    import piplite
    await piplite.install("ipywidgets")

In [ ]:
r"""**One transfer function, two realizations.** $G(z) = (\beta_0 z + \beta_1)/(z^2 + \alpha_1 z + \alpha_2)$
runs here as the direct form $m_k = \beta_0 e_{k-1} + \beta_1 e_{k-2} - \alpha_1 m_{k-1} - \alpha_2 m_{k-2}$
and as the state equations $x(k+1) = Ax(k) + Be(k)$, $m_k = Cx(k)$, both from rest with a unit step.
The defaults are the chapter's example, with $m_k = 4(1 - 0.5^k)^2 \to G(1) = 4$.
Move $\alpha_1$, $\alpha_2$ and watch the poles: complex poles make the steps ring, and a pole
outside the unit circle makes them grow. The two realizations always agree.
"""
import numpy as np
import matplotlib.pyplot as plt

plt.rcParams.update({"font.size": 11})
ORANGE, BLUE, TEAL, GRAY = "#E67E22", "#1D5FA8", "#1F6E78", "#6B7785"
NK = 21     # k = 0..20


def explore(alpha1=-0.75, alpha2=0.125, beta0=1.0, beta1=0.5):
    a1, a2, b0, b1 = (round(v, 10) for v in (alpha1, alpha2, beta0, beta1))
    e = np.ones(NK)
    # direct form
    m_dir = np.zeros(NK)
    for k in range(NK):
        em1 = e[k - 1] if k >= 1 else 0.0
        em2 = e[k - 2] if k >= 2 else 0.0
        mm1 = m_dir[k - 1] if k >= 1 else 0.0
        mm2 = m_dir[k - 2] if k >= 2 else 0.0
        m_dir[k] = b0 * em1 + b1 * em2 - a1 * mm1 - a2 * mm2
    # state equations (the two-delay form as matrices)
    A = np.array([[0.0, 1.0], [-a2, -a1]])
    B = np.array([0.0, 1.0])
    C = np.array([b1, b0])
    x = np.zeros(2)
    m_ss = np.zeros(NK)
    for k in range(NK):
        m_ss[k] = C @ x
        x = A @ x + B * e[k]

    poles = np.roots([1.0, a1, a2])
    rho = np.max(np.abs(poles))
    stable = rho < 1 - 1e-12
    G1 = (b0 + b1) / (1 + a1 + a2) if abs(1 + a1 + a2) > 1e-12 else np.inf

    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11.5, 4.0),
                                   gridspec_kw={"width_ratios": [1.9, 1]})
    k = np.arange(NK)
    ml, sl, bl = ax1.stem(k, m_dir, label="direct form")
    plt.setp(ml, color=ORANGE, markersize=6)
    plt.setp(sl, color=ORANGE, lw=2)
    plt.setp(bl, color="k", lw=0.6)
    ax1.plot(k, m_ss, "x", color=BLUE, ms=9, mew=1.6, label="state equations")
    if stable and np.isfinite(G1):
        ax1.axhline(G1, color=GRAY, ls="--", lw=1.2, label=f"G(1) = {G1:.3g}")
    ax1.set_xlabel("k")
    ax1.set_ylabel(r"$m_k$")
    ax1.set_title(rf"Unit step: $\alpha_1$ = {a1:g}, $\alpha_2$ = {a2:g}, "
                  rf"$\beta_0$ = {b0:g}, $\beta_1$ = {b1:g}", fontsize=11)
    ax1.legend(fontsize=10, loc="best")

    th = np.linspace(0, 2 * np.pi, 300)
    ax2.plot(np.cos(th), np.sin(th), color=GRAY, lw=1)
    ax2.axhline(0, color="k", lw=0.5)
    ax2.axvline(0, color="k", lw=0.5)
    ax2.plot(poles.real, poles.imag, "x", color=ORANGE, ms=12, mew=2.5, label="poles")
    if abs(b0) > 1e-12:
        ax2.plot([-b1 / b0], [0], "o", mfc="none", color=BLUE, ms=10, mew=1.8, label="zero")
    lim = max(1.3, 1.15 * rho)
    ax2.set_xlim(-lim, lim)
    ax2.set_ylim(-lim, lim)
    ax2.set_aspect("equal")
    ax2.set_xlabel("Re z")
    ax2.set_ylabel("Im z")
    ax2.set_title("poles of G(z)", fontsize=11)
    ax2.legend(fontsize=10, loc="upper left")
    fig.tight_layout()

    print("poles:", ", ".join(f"{p.real:.4g}{p.imag:+.4g}j" for p in poles),
          f"  max |z| = {rho:.4g}")
    print("m_0..m_5 =", np.round(m_dir[:6], 4),
          " (both realizations agree)" if np.allclose(m_dir, m_ss, rtol=1e-9, atol=1e-9) else "")
    if stable:
        print(f"stable: m_k -> G(1) = (beta0 + beta1)/(1 + alpha1 + alpha2) = {G1:.4g}")
    else:
        print("a pole on or outside the unit circle: m_k has no final value")
    plt.show()


try:
    from ipywidgets import interact, FloatSlider
    interact(explore,
             alpha1=FloatSlider(value=-0.75, min=-2.0, max=2.0, step=0.05, description="alpha1",
                                continuous_update=False),
             alpha2=FloatSlider(value=0.125, min=-1.2, max=1.2, step=0.025, description="alpha2",
                                continuous_update=False),
             beta0=FloatSlider(value=1.0, min=-2.0, max=2.0, step=0.1, description="beta0",
                               continuous_update=False),
             beta1=FloatSlider(value=0.5, min=-2.0, max=2.0, step=0.1, description="beta1",
                               continuous_update=False))
except ImportError:
    explore()

## Hands-on

**Hands-on, Chapter 5.** These cells go with the Hands-on exercises at the end of
Chapter 5. Do each exercise on paper first and write down your prediction; then use the
matching section below to check it. Each section has sliders: change one thing at a time.


In [ ]:
# In the browser (JupyterLite) the sliders need one extra package.
import sys
if "pyodide" in sys.modules:
    import piplite
    await piplite.install("ipywidgets")

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

plt.rcParams.update({"font.size": 11})
ORANGE, BLUE, TEAL, GRAY = "#E67E22", "#1D5FA8", "#1F6E78", "#6B7785"

In [ ]:
def run_state_space(A, B, C, D, e):
    """x(k+1) = A x(k) + B e(k), m(k) = C x(k) + D e(k), from x(0) = 0; returns m and the states."""
    x = np.zeros(A.shape[0])
    m, xs = [], []
    for ek in e:
        xs.append(x.copy())
        m.append(C @ x + D * ek)
        x = A @ x + B * ek
    return np.array(m), np.array(xs)

In [ ]:
# ---------------------------------------------------------------- H1
def h1_ring(T=0.05):
    """H1: e(t) = exp(-2t) cos(4 pi t) sampled every T; poles of E(z) at exp(-2T) exp(+-j 4 pi T)."""
    t = np.linspace(0, 3, 3000)
    k = np.arange(int(3 / T) + 1)
    r, th = np.exp(-2 * T), 4 * np.pi * T
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11.5, 3.8), gridspec_kw={"width_ratios": [2, 1]})
    ax1.plot(t, np.exp(-2 * t) * np.cos(4 * np.pi * t), color=GRAY, lw=1.2, label="e(t)")
    ax1.plot(k * T, np.exp(-2 * k * T) * np.cos(4 * np.pi * k * T), "o-", color=ORANGE, ms=5,
             lw=1, label="samples")
    ax1.axhline(0, color="k", lw=0.5)
    ax1.set_xlabel("t (s)")
    ax1.set_title(f"T = {T:g} s: angle {np.degrees(th):.1f} deg per sample "
                  f"({360 / np.degrees(th):.2f} samples per cycle), radius {r:.3f}", fontsize=10)
    ax1.legend(loc="upper right", fontsize=9)
    circ = np.linspace(0, 2 * np.pi, 300)
    ax2.plot(np.cos(circ), np.sin(circ), color=GRAY, lw=1)
    ax2.plot(r * np.cos(th) * np.ones(2), r * np.sin(th) * np.array([1, -1]), "x", color=ORANGE,
             ms=12, mew=2.5)
    ax2.set_aspect("equal")
    ax2.set_xlim(-1.3, 1.3)
    ax2.set_ylim(-1.3, 1.3)
    ax2.set_title("poles of E(z)", fontsize=11)
    plt.show()

In [ ]:
# ---------------------------------------------------------------- H2
def h2_programs():
    """H2: pulse response of D(z) = (2z^2 - 1.5z + 0.12)/(z^2 - z + 0.21): two programs."""
    A = np.array([[0, 1], [-0.21, 1.0]])
    B = np.array([0, 1.0])
    e = np.zeros(15)
    e[0] = 1
    m_good, _ = run_state_space(A, B, np.array([-0.3, 0.5]), 2.0, e)
    m_bad, _ = run_state_space(A, B, np.array([0.12, -1.5]), 0.0, e)
    # the difference equation itself: m(k) = m(k-1) - 0.21 m(k-2) + 2 e(k) - 1.5 e(k-1) + 0.12 e(k-2)
    m_de = np.zeros(15)
    for k in range(15):
        m_de[k] = ((m_de[k - 1] if k >= 1 else 0) - 0.21 * (m_de[k - 2] if k >= 2 else 0)
                   + 2 * e[k] - 1.5 * (e[k - 1] if k >= 1 else 0) + 0.12 * (e[k - 2] if k >= 2 else 0))
    k = np.arange(15)
    fig, ax = plt.subplots(figsize=(10, 3.6))
    ax.plot(k, m_de, "o", color=ORANGE, ms=9, mfc="none", mew=2, label="difference equation of D(z)")
    ax.plot(k, m_good, ".", color=BLUE, ms=8, label="corrected program")
    ax.plot(k, m_bad, "s--", color=TEAL, ms=4, lw=1, label="classmate's program")
    ax.axhline(0, color="k", lw=0.5)
    ax.set_xlabel("k")
    ax.set_title("pulse response: first samples " + ", ".join(f"{v:.3f}" for v in m_de[:3]),
                 fontsize=11)
    ax.legend(loc="lower right", fontsize=9)
    plt.show()

In [ ]:
# ---------------------------------------------------------------- H3
def h3_hidden_pole(c=1.0):
    """H3: G(z) = (z - c)/((z - 0.5)(z - 1.25)) in control canonical form, unit pulse input."""
    A = np.array([[0, 1], [-0.625, 1.75]])
    B = np.array([0, 1.0])
    n = 60
    e = np.zeros(n)
    e[0] = 1
    m, xs = run_state_space(A, B, np.array([-c, 1.0]), 0.0, e)
    k = np.arange(n)
    over = np.nonzero(np.abs(xs[:, 1]) > 32767)[0]
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11.5, 3.8))
    ax1.plot(k, m, "o-", color=ORANGE, ms=4, lw=1)
    ax1.set_ylim(-3, 3)
    ax1.axhline(0, color="k", lw=0.5)
    ax1.set_xlabel("k")
    ax1.set_title(f"c = {c:.3f}: output m(k)", fontsize=11)
    ax2.semilogy(k, np.abs(xs[:, 0]) + 1e-12, color=BLUE, lw=1.4, label="|x1|")
    ax2.semilogy(k, np.abs(xs[:, 1]) + 1e-12, "--", color=TEAL, lw=1.4, label="|x2|")
    ax2.axhline(32767, color=GRAY, ls=":", lw=1, label="16-bit limit")
    ax2.set_xlabel("k")
    ax2.set_ylim(1e-3, 1e6)
    ax2.set_title("stored numbers" + (f": x2 overflows at k = {over[0]}" if len(over) else ""),
                  fontsize=11)
    ax2.legend(loc="lower right", fontsize=9)
    plt.show()

In [ ]:
# ---------------------------------------------------------------- H4
def h4_hold(T_ms=0.3, held="start of period"):
    """H4: x(t) = 10 sin(2 pi t) mm held every T; gap |x - xbar| over one cycle."""
    T = T_ms * 1e-3
    n = int(np.ceil(1 / T))
    t = np.linspace(0, 1, 200001)
    k = np.floor(t / T)
    tk = (k + 0.5) * T if held.startswith("middle") else k * T
    gap = np.abs(10 * np.sin(2 * np.pi * t) - 10 * np.sin(2 * np.pi * tk))
    worst = gap.max()
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11.5, 3.8))
    win = t <= 12 * T
    ax1.plot(t[win] * 1e3, 10 * np.sin(2 * np.pi * t[win]), color=GRAY, lw=1.2, label="path x(t)")
    ax1.plot(t[win] * 1e3, 10 * np.sin(2 * np.pi * tk[win]), color=ORANGE, lw=1.6, label="held value")
    ax1.set_xlabel("t (ms)")
    ax1.set_ylabel("mm")
    ax1.set_title(f"first {12} periods ({n} updates per cycle)", fontsize=11)
    ax1.legend(loc="upper left", fontsize=9)
    ax2.plot(t, gap * 1e3, color=BLUE, lw=0.6)
    ax2.axhline(20, color=TEAL, ls="--", lw=1, label="spec 0.02 mm")
    ax2.set_xlabel("t (s)")
    ax2.set_ylabel("gap (micrometers)")
    ok = (worst <= 0.02) and (T_ms >= 0.5)
    ax2.set_title(f"{held}, T = {T_ms:.3f} ms: worst gap {worst * 1e3:.2f} um -> "
                  f"{'PASS' if ok else 'FAIL'}", fontsize=10)
    ax2.legend(loc="upper right", fontsize=9)
    plt.show()

In [ ]:
try:
    from ipywidgets import interact, FloatSlider, Dropdown
    interact(h1_ring, T=FloatSlider(value=0.05, min=0.01, max=0.6, step=0.01, description="T (s)",
                                    continuous_update=False))
    interact(h2_programs)
    interact(h3_hidden_pole, c=FloatSlider(value=1.0, min=0.0, max=1.5, step=0.01,
                                           description="c", continuous_update=False))
    interact(h4_hold, T_ms=FloatSlider(value=0.3, min=0.1, max=1.0, step=0.01, readout_format=".2f",
                                       description="T (ms)", continuous_update=False),
             held=Dropdown(options=["start of period", "middle of period"],
                           value="start of period", description="hold"))
except ImportError:
    h1_ring()
    h2_programs()
    h3_hidden_pole()
    h4_hold()